# Response-distance examples

The [README](../README.md) is the project overview; all proofs are in the
[technical appendix](../technical.md). This notebook only runs two small
examples. Use Python with NumPy and SciPy; no saved runs, ascent or GPU.
It works from repository root, this branch folder or this notebook folder.


## 1. A certified finite-scale corner


In [ ]:
import sys
from pathlib import Path

import numpy as np

repo = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "response_distance/code/response.py").is_file()
)
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

from response_distance.code.response_mixed import (
    planar_bv_logode_data,
    planar_merge_bounds,
    planar_merge_separation_radius,
)

path = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0]])
data = planar_bv_logode_data(path)
bounds = planar_merge_bounds(data, 0.05)
assert bounds.response_upper < 2e-5 < bounds.certificate_lower
{
    "joint_scale": 0.05,
    "response_error_upper": bounds.response_upper,
    "full_HS_certificate_lower": bounds.certificate_lower,
    "sufficient_scale_radius": planar_merge_separation_radius(data),
    "chosen_tolerance": 2e-5,
}

## 2. Storage movement and a Gram anchor


In [ ]:
import math
import sys
from pathlib import Path

import numpy as np

repo = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "response_distance/code/response.py").is_file()
)
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

from response_distance.code.response import (
    logode_merge_level_hs_norm,
    multi_logode_merge_level_hs_norm,
)
from response_distance.code.response_mixed import (
    planar_bv_logode_data,
    planar_merge_bounds,
)
from response_distance.code.response_storage import optimize_euler_storage

path = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0]])
bounds = planar_merge_bounds(planar_bv_logode_data(path), 0.05)
tolerance = 2e-5
assert bounds.response_upper < tolerance < bounds.certificate_lower

expected = 1 / math.sqrt(2)
assert math.isclose(
    logode_merge_level_hs_norm(1e100, 1e-100, 0.0, 2), expected, rel_tol=1e-13
)
assert math.isclose(
    multi_logode_merge_level_hs_norm(np.diag([1e200, 1e-200]), 2),
    expected,
    rel_tol=1e-13,
)

choices = {}
for label, criterion in [("exact_BV", "exact"), ("length_envelope", "fpl")]:
    choice, _ = optimize_euler_storage(2, 1.0, 0.01, criterion=criterion)
    choices[label] = {
        "N": choice.depth,
        "m": choice.intervals,
        "storage": choice.storage,
    }

{
    "planar_corner": {
        "joint_scale": 0.05,
        "response_upper": bounds.response_upper,
        "HS_lower": bounds.certificate_lower,
        "tolerance": tolerance,
    },
    "BV_storage": choices,
    "disparate_scale_Gram_anchor": "passed",
}